# 🚌 Analyse Exploratoire — Réseau de Transport Toulouse Métropole

**Auteur :** Étudiant Data Analyst  
**Source :** Open Data Toulouse Métropole  
**Objectif :** Comprendre la structure, la qualité et la distribution des arrêts du réseau Tisséo à travers une analyse statistique complète.

---

## Plan
1. Chargement et présentation des données
2. Qualité des données (valeurs manquantes, types)
3. Statistiques descriptives univariées
4. Analyses bivariées et corrélations
5. Focus : Stations VéloToulouse
6. Analyse géographique
7. Synthèse et recommandations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Blues_r')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

print('Bibliothèques chargées ✅')

---
## 1. Chargement et présentation des données

In [ ]:
# Chargement des 3 tables
df1 = pd.read_csv('../data/fichier1.csv', sep=';')  # Communes
df2 = pd.read_csv('../data/fichier2.csv', sep=';')  # Types de transport
df3 = pd.read_csv('../data/fichier3.csv', sep=';')  # Arrêts (table principale)

print(f'Communes      : {df1.shape[0]} lignes, {df1.shape[1]} colonnes')
print(f'Types         : {df2.shape[0]} lignes, {df2.shape[1]} colonnes')
print(f'Arrêts        : {df3.shape[0]} lignes, {df3.shape[1]} colonnes')

In [ ]:
print('=== Table Communes ==='); display(df1)
print('\n=== Types de transport ==='); display(df2)
print('\n=== Aperçu Arrêts ==='); display(df3.head(5))

In [ ]:
# Jointures : enrichissement de la table principale
df3['code_commune'] = df3['code_commune'].astype('Int64')
df3['code_type']    = df3['code_type'].astype('Int64')

df = (
    df3
    .merge(df1, left_on='code_commune', right_on='Code commune', how='left')
    .merge(df2, left_on='code_type',    right_on='Code type',    how='left')
)

print(f'Table enrichie : {df.shape[0]} lignes × {df.shape[1]} colonnes')
display(df[['nom', 'ligne', 'type', 'commune', 'nb_bornettes', 'nb_places']].head(8))

---
## 2. Qualité des données

> Avant toute analyse, évaluer la complétude des données est une étape critique en Data Analytics.

In [ ]:
missing = df3.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df3) * 100).round(1)

quality_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    '% manquant': missing_pct,
    'Complétude (%)': (100 - missing_pct)
}).query('`Valeurs manquantes` > 0')

display(quality_df.style
    .background_gradient(subset=['% manquant'], cmap='RdYlGn_r')
    .format({'% manquant': '{:.1f}%', 'Complétude (%)': '{:.1f}%'})
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E74C3C' if p > 50 else '#F39C12' if p > 20 else '#2ECC71'
          for p in quality_df['% manquant']]
bars = ax.barh(quality_df.index, quality_df['% manquant'], color=colors)
ax.axvline(50, color='red', linestyle='--', alpha=0.6, label='Seuil critique 50%')
ax.axvline(20, color='orange', linestyle='--', alpha=0.6, label='Seuil attention 20%')
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Complétude par colonne — Fichier arrêts', fontweight='bold', pad=12)
ax.invert_yaxis()
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\n💡 Insight : Les colonnes code_commune, en_service, couvert ont plus de 80% de données manquantes.')
print('   → Ces champs ne peuvent pas être utilisés seuls pour des analyses quantitatives fiables.')

---
## 3. Statistiques descriptives univariées

In [ ]:
# Distribution des types de transport
type_counts = df['type'].value_counts()
type_pct    = (type_counts / type_counts.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barres horizontales
palette = sns.color_palette('Blues_r', len(type_counts))
axes[0].barh(type_counts.index, type_counts.values, color=palette)
for i, (val, pct) in enumerate(zip(type_counts.values, type_pct.values)):
    axes[0].text(val + 5, i, f'{val} ({pct}%)', va='center', fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("Nombre d'arrêts")
axes[0].set_title('Distribution par type de transport', fontweight='bold')
axes[0].spines[['top', 'right']].set_visible(False)

# Camembert (top 6 + autres)
top6 = type_counts.head(6)
other = type_counts.iloc[6:].sum()
pie_data = pd.concat([top6, pd.Series({'Autres': other})])
axes[1].pie(pie_data.values, labels=pie_data.index, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('Set2', len(pie_data)),
            pctdistance=0.82)
axes[1].set_title('Part modale (top 6 + autres)', fontweight='bold')

plt.suptitle("Répartition des arrêts par type de transport", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'\n💡 Le bus représente {type_pct["bus"]:.1f}% des arrêts, soit le mode dominant du réseau.')

In [ ]:
# Arrêts par commune
commune_counts = df['commune'].value_counts()
total_known    = commune_counts.sum()
total_missing  = df['commune'].isna().sum()

print(f'Arrêts avec commune renseignée : {total_known} ({total_known/len(df)*100:.1f}%)')
print(f'Arrêts sans commune            : {total_missing} ({total_missing/len(df)*100:.1f}%)')
print()
display(commune_counts.rename('Nombre d\'arrêts').to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(commune_counts.index, commune_counts.values,
              color=sns.color_palette('Paired', len(commune_counts)))
for bar, val in zip(bars, commune_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(val), ha='center', fontsize=9)
ax.set_ylabel("Nombre d'arrêts")
ax.set_title('Arrêts par commune (données renseignées uniquement)', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

print('\n⚠️  Biais important : 81% des arrêts n\'ont pas de commune renseignée.')
print('   → Les analyses par commune sont à interpréter avec précaution.')

In [ ]:
# Top 20 lignes
top_lignes = df3['ligne'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(top_lignes.index.astype(str), top_lignes.values,
       color=sns.color_palette('viridis', len(top_lignes)))
ax.set_xlabel('Ligne')
ax.set_ylabel("Nombre d'arrêts")
ax.set_title('Top 20 des lignes par nombre d\'arrêts', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'\n💡 La ligne 119 est la plus dense avec {top_lignes.iloc[0]} arrêts.')

---
## 4. Analyses bivariées

In [ ]:
# Répartition type × commune (arrêts renseignés seulement)
df_known = df.dropna(subset=['commune', 'type'])
ct = pd.crosstab(df_known['commune'], df_known['type'])

# Top 8 types pour lisibilité
top_types = df_known['type'].value_counts().head(8).index
ct_top    = ct[top_types]

fig, ax = plt.subplots(figsize=(12, 5))
ct_top.plot(kind='bar', stacked=True, ax=ax,
            colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Commune')
ax.set_ylabel("Nombre d'arrêts")
ax.set_title('Répartition des types de transport par commune', fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=20, ha='right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\n💡 Toulouse concentre tous les modes lourds (métro, tramway).')
print('   Les communes périphériques dépendent quasi-exclusivement du bus et des lignes affrétées.')

In [ ]:
# Heatmap normalisée (proportion par commune)
ct_pct = ct_top.div(ct_top.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(ct_pct.round(1), annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% des arrêts de la commune'})
ax.set_title('Part modale par commune (% des arrêts renseignés)', fontweight='bold')
ax.set_xlabel('Type de transport')
ax.set_ylabel('Commune')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('\n💡 La heatmap révèle des profils de mobilité très différents selon les communes.')

---
## 5. Focus : Stations VéloToulouse

> VéloToulouse est le service de vélos en libre-service de la métropole. Avec 284 stations, c'est un dataset riche pour une analyse de capacité.

In [ ]:
velo = df[df['type'] == 'VeloToulouse'][['nom', 'nb_bornettes', 'commune']].copy()
velo = velo.dropna(subset=['nb_bornettes'])
velo['nb_bornettes'] = velo['nb_bornettes'].astype(int)

print(f'Stations avec données de capacité : {len(velo)}')
print()
stats = velo['nb_bornettes'].describe()
print(stats.round(2))
print(f'\nCoefficient de variation : {(stats["std"] / stats["mean"] * 100):.1f}%')
print(f'Asymétrie (skewness)     : {velo["nb_bornettes"].skew():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogramme
axes[0].hist(velo['nb_bornettes'], bins=15, color='#4C9BE8', edgecolor='white')
axes[0].axvline(velo['nb_bornettes'].mean(),   color='red',    linestyle='--',
                label=f'Moyenne : {velo["nb_bornettes"].mean():.1f}')
axes[0].axvline(velo['nb_bornettes'].median(), color='orange', linestyle='--',
                label=f'Médiane : {velo["nb_bornettes"].median():.0f}')
axes[0].set_xlabel('Bornettes')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution du nb de bornettes', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Box plot
axes[1].boxplot(velo['nb_bornettes'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#4C9BE8', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('Nombre de bornettes')
axes[1].set_title('Box plot — Capacité des stations', fontweight='bold')
axes[1].set_xticks([])
axes[1].spines[['top', 'right']].set_visible(False)

# Catégories de capacité
bins_labels = ['Petite\n(<10)', 'Moyenne\n(10-19)', 'Grande\n(≥20)']
bins_counts = [
    (velo['nb_bornettes'] < 10).sum(),
    ((velo['nb_bornettes'] >= 10) & (velo['nb_bornettes'] < 20)).sum(),
    (velo['nb_bornettes'] >= 20).sum()
]
bars = axes[2].bar(bins_labels, bins_counts,
                   color=['#E74C3C', '#F39C12', '#2ECC71'], edgecolor='white')
for bar, val in zip(bars, bins_counts):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val}\n({val/len(velo)*100:.0f}%)', ha='center', fontsize=9)
axes[2].set_ylabel('Nombre de stations')
axes[2].set_title('Catégories de capacité', fontweight='bold')
axes[2].spines[['top', 'right']].set_visible(False)

plt.suptitle('Analyse de la capacité des stations VéloToulouse', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'\n💡 {bins_counts[2]} stations ({bins_counts[2]/len(velo)*100:.0f}%) ont une grande capacité (≥20 bornettes).')
print(f'   La distribution est concentrée autour de 20 bornettes (valeur modale).')

In [ ]:
# Top et flop stations
print('🏆 Top 10 stations les plus capacitaires :')
display(velo.nlargest(10, 'nb_bornettes')[['nom', 'nb_bornettes']].reset_index(drop=True))

print('\n⚠️  Stations avec le moins de bornettes (≤ 5) :')
display(velo[velo['nb_bornettes'] <= 5][['nom', 'nb_bornettes']].reset_index(drop=True))

---
## 6. Analyse géographique

In [ ]:
# Extraction des coordonnées
geo = df.copy()
geo[['lat', 'lon']] = geo['geo point'].str.split(',', expand=True).astype(float)
geo = geo.dropna(subset=['lat', 'lon', 'type'])

# Scatter plot géographique coloré par type (top 6)
top6_types  = df['type'].value_counts().head(6).index
geo_top6    = geo[geo['type'].isin(top6_types)]
palette_map = dict(zip(top6_types, sns.color_palette('tab10', 6)))

fig, ax = plt.subplots(figsize=(10, 9))
for transport_type, group in geo_top6.groupby('type'):
    ax.scatter(group['lon'], group['lat'],
               label=transport_type,
               color=palette_map[transport_type],
               alpha=0.5, s=8)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Répartition géographique des arrêts par type de transport\n(Toulouse Métropole)',
             fontweight='bold')
ax.legend(title='Type', markerscale=2, fontsize=9)
ax.set_aspect('equal')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\n💡 La répartition géographique montre une concentration des arrêts bus en périphérie')
print('   et des stations VéloToulouse densément regroupées dans le centre de Toulouse.')

In [ ]:
# Densité 2D (KDE) — tous types confondus
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# KDE global
sns.kdeplot(data=geo, x='lon', y='lat', fill=True, cmap='YlOrRd',
            thresh=0.05, levels=12, ax=axes[0])
axes[0].set_title('Densité de tous les arrêts (KDE)', fontweight='bold')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_aspect('equal')

# KDE VéloToulouse seulement
geo_velo = geo[geo['type'] == 'VeloToulouse']
sns.kdeplot(data=geo_velo, x='lon', y='lat', fill=True, cmap='Blues',
            thresh=0.05, levels=12, ax=axes[1])
axes[1].set_title('Densité des stations VéloToulouse (KDE)', fontweight='bold')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_aspect('equal')

plt.suptitle('Cartes de densité — Réseau Tisséo', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Synthèse et recommandations

### 📌 Ce que nous avons appris

In [ ]:
summary = pd.DataFrame({
    'Indicateur': [
        'Total arrêts et stations',
        'Types de transport distincts',
        'Arrêts bus (mode principal)',
        'Stations VéloToulouse',
        'Moy. bornettes / station vélo',
        'Arrêts sans commune renseignée',
        'Lignes distinctes',
        'Stations métro',
        'Stations tramway',
    ],
    'Valeur': [
        f"{len(df3):,}",
        df2.shape[0],
        f"{df['type'].value_counts()['bus']:,} ({df['type'].value_counts()['bus']/len(df)*100:.1f}%)",
        df[df['type']=='VeloToulouse'].shape[0],
        f"{df['nb_bornettes'].mean():.1f}",
        f"{df3['code_commune'].isna().sum():,} ({df3['code_commune'].isna().sum()/len(df3)*100:.1f}%)",
        df3['ligne'].nunique(),
        df[df['type']=='Metro'].shape[0],
        df[df['type']=='Tramway'].shape[0],
    ]
})
display(summary.style.set_properties(**{'text-align': 'left'}))

### 🔎 Recommandations Data Quality

1. **Compléter le champ `code_commune`** pour 81% des arrêts — nécessaire pour toute analyse territoriale fiable.
2. **Normaliser `en_service`** : le champ mélange des valeurs booléennes (`oui/non`) et des années (2010, 2013…), ce qui rend toute agrégation impossible sans nettoyage.
3. **Vérifier les stations VéloToulouse sans bornettes** (nb_bornettes = 0) : sont-elles en travaux ou des erreurs de saisie ?

### 📈 Pistes d'analyse complémentaires

- **Accessibilité** : croiser avec des données de population pour calculer un ratio habitants / arrêts par commune.
- **Temporal** : exploiter le champ `en_service` (années) pour une analyse de l'évolution du réseau dans le temps.
- **Optimisation** : identifier les zones géographiques sous-desservies à partir de la densité KDE.

---
*Notebook réalisé dans le cadre d'un projet Data Analyst étudiant.*